# 45 — Avaliação OOD: Fake.Br + FakeRecogna em todos os modelos

Aplica os modelos treinados (BERT, TF-IDF — binário e multiclasse) sobre dois corpora externos.

| Dataset | Positivo binário | Nível 3 |
|---|---|---|
| **Fake.Br** (Lira et al.) | `metadata_category == 'economia'` (44/7200) | Subgrupo por veracidade (fake vs true) |
| **FakeRecogna** | URL do portal `economia.uol.com.br` (312/11872) | Corte balanceado intra-UOL (312 economia + 312 outros UOL real, seed=2026) |

Cada nível emite `result_card.json` em `<DRIVE>/ood_runs/<model_id>_<task>_<dataset>_<level>/`. A célula final agrega cards e roda McNemar pareado por `(domain, level)`.

**Pré-requisitos**:
- `colab_ood_data.zip` em `<DRIVE>/economy-classifier/colab_ood_data.zip` (vide seção 3 para layout).
- Modelos treinados em `<DRIVE>/economy-classifier/runs/<model_id>_<task>_test_set/model/` (HF dir para BERT, `tfidf_pipeline.joblib` para TF-IDF), gerados pelos NBs 21 / 11 / 12 / 13.


## 0. Verificação de ambiente

In [ ]:
import torch

if not torch.cuda.is_available():
    print("AVISO: GPU nao detectada. BERT vai rodar em CPU (lento). "
          "Para acelerar: Runtime > Change runtime type > GPU.")
    GPU_NAME = "CPU"
    HARDWARE = "Colab-CPU"
else:
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f"GPU: {GPU_NAME} ({VRAM_GB} GB VRAM)")
    HARDWARE = f"Colab-{GPU_NAME.split()[-1]}"
print("CUDA:", torch.version.cuda)


## 1. Bootstrap (Colab + local)

In [ ]:
import subprocess
import sys
import zipfile
from pathlib import Path


def _run(cmd: list[str], description: str) -> None:
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr, file=sys.stderr)
        raise RuntimeError(f"{description} failed with exit code {result.returncode}")


IN_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")
print("Python   :", sys.version.split()[0])

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/almeidadm/economy-classifier.git"
    REPO_BRANCH = "main"
    DRIVE_FOLDER = "economy-classifier"

    DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/economy-classifier")

    if REPO_DIR.exists():
        _run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], "git fetch")
        _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], "git reset")
    else:
        _run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], "git clone")

    _run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR),
         "--upgrade-strategy", "only-if-needed", "-q"],
        "pip install -e .",
    )

    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

    RUNS_BASE = DRIVE_BASE / "runs"
    OOD_RUNS_BASE = DRIVE_BASE / "ood_runs"
    OOD_DATA_DIR = Path("/content/ood_data")
else:
    REPO_DIR = Path.cwd().parent
    DRIVE_BASE = REPO_DIR / "artifacts"
    RUNS_BASE = DRIVE_BASE / "runs"
    OOD_RUNS_BASE = DRIVE_BASE / "ood_runs"
    OOD_DATA_DIR = REPO_DIR / "ood_data"

OOD_RUNS_BASE.mkdir(parents=True, exist_ok=True)
OOD_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_DIR     :", REPO_DIR)
print("RUNS_BASE    :", RUNS_BASE)
print("OOD_RUNS_BASE:", OOD_RUNS_BASE)
print("OOD_DATA_DIR :", OOD_DATA_DIR)


## 2. Imports dos scripts de avaliação

In [ ]:
# scripts/ nao e um pacote Python — adicionamos manualmente ao path
SCRIPTS_DIR = REPO_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import evaluate_fake_br as fb
import evaluate_fake_recogna as fr
from economy_classifier.project import compute_artifact_size_mb
from economy_classifier.evaluation import compute_mcnemar_pairwise

FAKE_BR_ROOT = OOD_DATA_DIR / "fake_br"
FAKE_RECOGNA_ROOT = OOD_DATA_DIR / "fake_recogna"

print("FAKE_BR_ROOT   :", FAKE_BR_ROOT)
print("FAKE_RECOGNA   :", FAKE_RECOGNA_ROOT)


## 3. Carregamento dos datasets OOD

**Layout esperado** dentro de `colab_ood_data.zip`:

```
fake_br/
  full_texts/
    fake/<id>.txt
    true/<id>.txt
    fake-meta-information/<id>-meta.txt
    true-meta-information/<id>-meta.txt
fake_recogna/
  FakeRecogna_*.xlsx
```

**Para gerar o zip localmente** (uma vez, antes do primeiro upload):

```bash
cd ~/Documentos/repositorios/fn-dataset-eda/data/raw
zip -r ~/Documentos/repositorios/economy-classifier/colab_ood_data.zip fake_br fake_recogna
# Suba colab_ood_data.zip para <DRIVE>/economy-classifier/
```

Em execução **local**, a célula abaixo cria links simbólicos para o repo `fn-dataset-eda` vizinho.


In [ ]:
if IN_COLAB:
    zip_path = DRIVE_BASE / "colab_ood_data.zip"
    if not (FAKE_BR_ROOT.exists() and FAKE_RECOGNA_ROOT.exists()):
        assert zip_path.exists(), (
            f"Falta {zip_path}. Veja a secao 3 acima para gerar o zip "
            "localmente e fazer upload para o Drive."
        )
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(OOD_DATA_DIR)
        print(f"Extraido {zip_path.name} -> {OOD_DATA_DIR}")
else:
    LOCAL_FN_REPO = Path("/home/diacrono/Documentos/repositorios/fn-dataset-eda/data/raw")
    if not FAKE_BR_ROOT.exists() and (LOCAL_FN_REPO / "fake_br").exists():
        FAKE_BR_ROOT.symlink_to(LOCAL_FN_REPO / "fake_br")
    if not FAKE_RECOGNA_ROOT.exists() and (LOCAL_FN_REPO / "fake_recogna").exists():
        FAKE_RECOGNA_ROOT.symlink_to(LOCAL_FN_REPO / "fake_recogna")

assert FAKE_BR_ROOT.exists(), f"Fake.Br ausente em {FAKE_BR_ROOT}"
assert FAKE_RECOGNA_ROOT.exists(), f"FakeRecogna ausente em {FAKE_RECOGNA_ROOT}"

print("Carregando Fake.Br...")
fb_df = fb.load_fake_br(FAKE_BR_ROOT)
print(f"  {len(fb_df)} docs | economia={int((fb_df.y_true_binary==1).sum())}")

print("Carregando FakeRecogna...")
fr_df = fr.load_fake_recogna(FAKE_RECOGNA_ROOT)
print(f"  {len(fr_df)} docs | economia.uol={int((fr_df.y_true_binary==1).sum())}")

# Caches reaproveitados em todos os modelos
fb_texts = fb_df["text"].fillna("").tolist()
fr_texts = fr_df["text"].fillna("").tolist()


## 4. Descoberta dos modelos no Drive

Procura por `<RUNS_BASE>/<model_id>_<task>_test_set/model/` (convenção dos NBs 21 / 11 / 12 / 13). LLMs e ensembles ficam de fora — exigem orquestração extra.


In [ ]:
import re
import pandas as pd

RUN_PATTERN = re.compile(r"^(bert|tfidf)_.+_(binary|multiclass)_test_set$")

discovered = []
if not RUNS_BASE.exists():
    print(f"AVISO: RUNS_BASE nao existe: {RUNS_BASE}")
else:
    for run_dir in sorted(RUNS_BASE.iterdir()):
        if not run_dir.is_dir():
            continue
        m = RUN_PATTERN.match(run_dir.name)
        if not m:
            continue
        model_dir = run_dir / "model"
        if not model_dir.is_dir():
            # Sem pesos persistidos no Drive — pular silenciosamente
            continue
        try:
            model_type = fb.detect_model_type(model_dir)
        except FileNotFoundError as exc:
            print(f"  pula {run_dir.name}: {exc}")
            continue
        task = m.group(2)
        suffix = f"_{task}_test_set"
        model_id = run_dir.name[: run_dir.name.rfind(suffix)]
        discovered.append({
            "model_id": model_id,
            "task": task,
            "model_type": model_type,
            "model_dir": str(model_dir),
            "run_dir": str(run_dir),
        })

models_df = pd.DataFrame(discovered)
print(f"\n{len(models_df)} modelos com pesos no Drive:")
display(models_df[["model_id", "task", "model_type"]] if len(models_df) else models_df)
assert len(models_df) > 0, (
    f"Nenhum modelo encontrado em {RUNS_BASE}. Verifique se os NBs 21/11/12/13 "
    "salvaram os pesos no Drive (subdir 'model/' dentro de cada run)."
)


## 5. Avaliação em loop

Para cada modelo: 1 inferência por dataset (não 1 por nível). Os helpers `evaluate_level{1,2,3}_*` consomem o mesmo array `probs` para emitir cards independentes.


In [ ]:
import time

DEFAULT_BATCH = 64
DEFAULT_MAX_LEN = 128


def run_inference(model_dir, model_type, texts):
    if model_type == "bert":
        return fb.predict_bert(
            texts, model_dir,
            batch_size=DEFAULT_BATCH, max_length=DEFAULT_MAX_LEN,
        )
    return fb.predict_tfidf(texts, model_dir)


run_log = []
for _, row in models_df.iterrows():
    model_dir = Path(row["model_dir"])
    model_id = row["model_id"]
    task = row["task"]
    model_type = row["model_type"]
    print(f"\n=== {model_id} | task={task} | type={model_type} ===")

    model_size_mb = round(compute_artifact_size_mb(model_dir), 3)
    expected_classes = 2 if task == "binary" else 8
    t0 = time.perf_counter()

    # --- Fake.Br ---
    try:
        fb_probs, fb_classes, fb_inf_s, fb_info = run_inference(model_dir, model_type, fb_texts)
    except Exception as exc:  # noqa: BLE001
        print(f"  ERRO Fake.Br inferencia: {type(exc).__name__}: {exc}")
        continue
    if fb_probs.shape[1] != expected_classes:
        print(f"  AVISO: modelo emite {fb_probs.shape[1]} classes; esperado {expected_classes}. Pulando.")
        continue
    print(f"  Fake.Br  inf: {fb_inf_s:.1f}s, shape={fb_probs.shape}")

    common_fb = dict(
        df=fb_df, probs=fb_probs, classes=fb_classes,
        model_id=model_id, model_type=model_type,
        output_root=OOD_RUNS_BASE, inference_seconds=fb_inf_s,
        model_size_mb=model_size_mb, max_length=DEFAULT_MAX_LEN,
        n_parameters=fb_info["n_parameters"], hardware=fb_info["hardware"],
    )
    if task == "binary":
        fb.evaluate_level1_binary(**common_fb)
        fb.evaluate_level3_subgroup(**common_fb)
    else:
        fb.evaluate_level2_multiclass(**common_fb)

    # --- FakeRecogna ---
    try:
        fr_probs, fr_classes, fr_inf_s, fr_info = run_inference(model_dir, model_type, fr_texts)
    except Exception as exc:  # noqa: BLE001
        print(f"  ERRO FakeRecogna inferencia: {type(exc).__name__}: {exc}")
        continue
    print(f"  FakeReco inf: {fr_inf_s:.1f}s, shape={fr_probs.shape}")

    common_fr = dict(
        df=fr_df, probs=fr_probs, classes=fr_classes,
        model_id=model_id, model_type=model_type,
        output_root=OOD_RUNS_BASE, inference_seconds=fr_inf_s,
        model_size_mb=model_size_mb, max_length=DEFAULT_MAX_LEN,
        n_parameters=fr_info["n_parameters"], hardware=fr_info["hardware"],
    )
    if task == "binary":
        fr.evaluate_level1_binary(**common_fr)
        fr.evaluate_level3_uol_balanced(**common_fr, task="binary")
    else:
        fr.evaluate_level2_multiclass(**common_fr)
        fr.evaluate_level3_uol_balanced(**common_fr, task="multiclass")

    run_log.append({
        "model_id": model_id, "task": task,
        "fb_inference_s": round(fb_inf_s, 2),
        "fr_inference_s": round(fr_inf_s, 2),
        "total_s": round(time.perf_counter() - t0, 2),
    })

print("\n=== RUN LOG ===")
display(pd.DataFrame(run_log))


## 6. Agregação dos cards OOD

Walk em `OOD_RUNS_BASE/*/result_card.json` filtrando por `config.domain in {fake_br_full_texts, fake_recogna_economia_uol}`.


In [ ]:
import json

OOD_DOMAINS = {"fake_br_full_texts", "fake_recogna_economia_uol"}

rows = []
for card_path in sorted(OOD_RUNS_BASE.glob("*/result_card.json")):
    card = json.loads(card_path.read_text())
    domain = card.get("config", {}).get("domain")
    if domain not in OOD_DOMAINS:
        continue
    metrics = card.get("metrics", {})
    rows.append({
        "model_id": card.get("model_id"),
        "task": card.get("task"),
        "domain": domain,
        "level": card.get("config", {}).get("level"),
        "n_eval": card.get("n_eval_samples"),
        # Binarias (se aplicaveis)
        "f1": metrics.get("f1"),
        "precision": metrics.get("precision"),
        "recall": metrics.get("recall"),
        "auc_roc": metrics.get("auc_roc"),
        "brier": metrics.get("brier"),
        "ece": metrics.get("ece"),
        "positive_prevalence": metrics.get("positive_prevalence"),
        # Multiclasse
        "macro_f1": metrics.get("macro_f1"),
        "macro_f1_present_only": metrics.get("macro_f1_present_only"),
        "weighted_f1": metrics.get("weighted_f1"),
        "accuracy": metrics.get("accuracy"),
        "card_path": str(card_path.relative_to(OOD_RUNS_BASE)),
    })

cards_df = pd.DataFrame(rows).sort_values(["domain", "task", "level", "model_id"]).reset_index(drop=True)
print(f"{len(cards_df)} cards OOD agregados\n")
display(cards_df)


### 6.1 Pivot: F1 binário por (modelo × nível)

In [ ]:
if len(cards_df):
    pivot_bin = (
        cards_df[cards_df.task == "binary"]
        .pivot_table(index="model_id", columns=["domain", "level"], values="f1")
        .round(4)
    )
    print("F1 binario:")
    display(pivot_bin)

    pivot_macro = (
        cards_df[cards_df.task == "multiclass"]
        .pivot_table(index="model_id", columns=["domain", "level"], values="macro_f1")
        .round(4)
    )
    print("\nMacro-F1 multiclasse:")
    display(pivot_macro)


## 7. McNemar pareado por (domain, level)

Apenas para o nível **binário** (McNemar é teste 2×2). Bonferroni aplicado automaticamente sobre `K*(K-1)/2` pares dentro de cada grupo.


In [ ]:
from collections import defaultdict

groups: dict[tuple, dict] = defaultdict(dict)
for card_path in sorted(OOD_RUNS_BASE.glob("*/result_card.json")):
    card = json.loads(card_path.read_text())
    if card.get("config", {}).get("domain") not in OOD_DOMAINS:
        continue
    if card.get("task") != "binary":
        continue
    pred_path = card_path.parent / "predictions.csv"
    if not pred_path.exists():
        continue
    pred_df = pd.read_csv(pred_path)
    if "index" not in pred_df.columns or "y_pred" not in pred_df.columns:
        continue
    key = (card["config"]["domain"], card["config"]["level"], card["task"])
    groups[key][card["model_id"]] = pred_df.set_index("index")

mcnemar_results: dict[tuple, pd.DataFrame] = {}
for (domain, level, task), preds_by_model in sorted(groups.items()):
    if len(preds_by_model) < 2:
        continue
    common_idx = None
    for df in preds_by_model.values():
        common_idx = df.index if common_idx is None else common_idx.intersection(df.index)
    aligned = {m: df.loc[common_idx, "y_pred"].to_numpy() for m, df in preds_by_model.items()}
    y_true = next(iter(preds_by_model.values())).loc[common_idx, "y_true"].to_numpy()

    pw = compute_mcnemar_pairwise(y_true, aligned)
    mcnemar_results[(domain, level, task)] = pw
    print(f"\n=== domain={domain} | level={level} | task={task} ===")
    print(f"  n={len(common_idx)}, k_modelos={len(aligned)}, n_pares={len(pw)}")
    display(pw[["method_a", "method_b", "p_value", "p_value_adjusted", "significant_after_correction"]])


## 8. Export para `<DRIVE>/reports/ood_evaluation/`

In [ ]:
REPORT_DIR = OOD_RUNS_BASE.parent / "reports" / "ood_evaluation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

cards_df.to_csv(REPORT_DIR / "ood_cards_table.csv", index=False)
print(f"Tabela de cards: {REPORT_DIR / 'ood_cards_table.csv'}")

if len(cards_df):
    if (cards_df.task == "binary").any():
        cards_df[cards_df.task == "binary"].pivot_table(
            index="model_id", columns=["domain", "level"], values="f1",
        ).round(4).to_csv(REPORT_DIR / "ood_pivot_binary_f1.csv")
        print(f"Pivot F1 binario: {REPORT_DIR / 'ood_pivot_binary_f1.csv'}")
    if (cards_df.task == "multiclass").any():
        cards_df[cards_df.task == "multiclass"].pivot_table(
            index="model_id", columns=["domain", "level"], values="macro_f1",
        ).round(4).to_csv(REPORT_DIR / "ood_pivot_multiclass_macrof1.csv")
        print(f"Pivot macro-F1 multi: {REPORT_DIR / 'ood_pivot_multiclass_macrof1.csv'}")

for (domain, level, task), pw in mcnemar_results.items():
    fname = f"mcnemar_{domain}_{level}_{task}.csv"
    pw.to_csv(REPORT_DIR / fname, index=False)
    print(f"McNemar: {REPORT_DIR / fname}")

print(f"\nDestino: {REPORT_DIR}")
